# Experiment No. 5 — DCGAN for Image Generation


## Step 1: Import Libraries

In [4]:
!pip install tensorflow

In [6]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt
import os

print('TensorFlow version:', tf.__version__)
print('GPU Available:', tf.config.list_physical_devices('GPU'))

os.makedirs('generated_images', exist_ok=True)

ImportError: Traceback (most recent call last):
  File "C:\aaa\Lib\site-packages\tensorflow\python\pywrap_tensorflow.py", line 73, in <module>
    from tensorflow.python._pywrap_tensorflow_internal import *
ImportError: DLL load failed while importing _pywrap_tensorflow_internal: A dynamic link library (DLL) initialization routine failed.


Failed to load the native TensorFlow runtime.
See https://www.tensorflow.org/install/errors for some common causes and solutions.
If you need help, create an issue at https://github.com/tensorflow/tensorflow/issues and include the entire stack trace above this error message.

## Step 2: Load and Preprocess MNIST Dataset
- Images are normalized from `[0, 255]` to `[-1, 1]` to match the Generator's `tanh` output range.
- Dataset is shuffled and batched for efficient training.

In [26]:
(X_train, _), (_, _) = tf.keras.datasets.mnist.load_data()

# Normalize pixel values from [0,255] to [-1,1]
X_train = (X_train.astype('float32') - 127.5) / 127.5
X_train = np.expand_dims(X_train, axis=-1)  # (60000, 28, 28, 1)

BUFFER_SIZE = 60000
BATCH_SIZE  = 64
NOISE_DIM   = 100

dataset = tf.data.Dataset.from_tensor_slices(X_train).shuffle(BUFFER_SIZE).batch(BATCH_SIZE)

print(f'Training samples : {X_train.shape[0]}')
print(f'Image shape      : {X_train.shape[1:]}')
print(f'Pixel range      : [{X_train.min():.1f}, {X_train.max():.1f}]')
print(f'Batch size       : {BATCH_SIZE}')
print(f'Batches per epoch: {len(dataset)}')

NameError: name 'tf' is not defined

## Step 3: Visualize Sample Real Images

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
fig.suptitle('Sample Real MNIST Images', fontsize=14, fontweight='bold')
for i, ax in enumerate(axes.flatten()):
    ax.imshow(X_train[i, :, :, 0], cmap='gray')
    ax.axis('off')
plt.tight_layout()
plt.show()

## Step 4: Build the Generator

The Generator takes a **random noise vector of size 100** and upsamples it into a **28×28 grayscale image** using:
- `Dense` → `Reshape` to produce a 7×7 feature map
- `Conv2DTranspose` layers to progressively upsample: 7×7 → 14×14 → 28×28
- `BatchNormalization` for training stability
- `LeakyReLU` to prevent dying neurons
- Final `tanh` activation outputs values in `[-1, 1]`

In [ ]:
def build_generator():
    model = models.Sequential(name='Generator')

    # Dense + Reshape: noise (100,) → feature map (7, 7, 256)
    model.add(layers.Dense(7 * 7 * 256, use_bias=False, input_shape=(NOISE_DIM,)))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())
    model.add(layers.Reshape((7, 7, 256)))

    # Conv2DTranspose: 7×7 → 7×7  (stride=1)
    model.add(layers.Conv2DTranspose(128, (5, 5), strides=(1, 1), padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    # Conv2DTranspose: 7×7 → 14×14  (stride=2)
    model.add(layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    # Conv2DTranspose: 14×14 → 28×28  (stride=2), 1 output channel
    model.add(layers.Conv2DTranspose(1, (5, 5), strides=(2, 2), padding='same',
                                     use_bias=False, activation='tanh'))
    return model

generator = build_generator()
generator.summary()

## Step 5: Build the Discriminator

The Discriminator is a **binary classifier** that takes a 28×28 image and outputs a single logit:
- `Conv2D` with stride 2 to downsample: 28×28 → 14×14 → 7×7
- `LeakyReLU` for better gradient flow
- `Dropout(0.3)` to prevent overfitting
- `Flatten` + `Dense(1)` outputs a raw logit (real or fake)

In [ ]:
def build_discriminator():
    model = models.Sequential(name='Discriminator')

    # Conv2D: 28×28 → 14×14
    model.add(layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same', input_shape=(28, 28, 1)))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    # Conv2D: 14×14 → 7×7
    model.add(layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same'))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    # Classify
    model.add(layers.Flatten())
    model.add(layers.Dense(1))  # raw logit — sigmoid applied inside loss (from_logits=True)

    return model

discriminator = build_discriminator()
discriminator.summary()

## Step 6: Define Loss Functions and Optimizers

| | Target Label | Formula |
|---|---|---|
| **Discriminator (real)** | 1 | BCE(1, D(real)) |
| **Discriminator (fake)** | 0 | BCE(0, D(G(z))) |
| **Generator** | 1 (fool D) | BCE(1, D(G(z))) |

Both networks use **Adam optimizer** with learning rate `1e-4`.

In [6]:
cross_entropy  = tf.keras.losses.BinaryCrossentropy(from_logits=True)
gen_optimizer  = tf.keras.optimizers.Adam(1e-4)
disc_optimizer = tf.keras.optimizers.Adam(1e-4)

def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output),  real_output)   # real images → 1
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)   # fake images → 0
    return real_loss + fake_loss

def generator_loss(fake_output):
    # Generator wants discriminator to classify fakes as real (→ 1)
    return cross_entropy(tf.ones_like(fake_output), fake_output)

print('Loss functions and optimizers ready.')

NameError: name 'tf' is not defined

## Step 7: Define the Training Step

Per step:
1. Sample noise → generate fake images
2. Update **Discriminator** on real + fake batches
3. Update **Generator** to fool the updated Discriminator

`@tf.function` traces the function as a TF graph for faster GPU execution.

In [ ]:
@tf.function
def train_step(images):
    noise = tf.random.normal([BATCH_SIZE, NOISE_DIM])

    # --- Discriminator update ---
    with tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)
        real_output      = discriminator(images,           training=True)
        fake_output      = discriminator(generated_images, training=True)
        d_loss           = discriminator_loss(real_output, fake_output)

    grads_disc = disc_tape.gradient(d_loss, discriminator.trainable_variables)
    disc_optimizer.apply_gradients(zip(grads_disc, discriminator.trainable_variables))

    # --- Generator update ---
    noise = tf.random.normal([BATCH_SIZE, NOISE_DIM])
    with tf.GradientTape() as gen_tape:
        generated_images = generator(noise, training=True)
        fake_output      = discriminator(generated_images, training=True)
        g_loss           = generator_loss(fake_output)

    grads_gen = gen_tape.gradient(g_loss, generator.trainable_variables)
    gen_optimizer.apply_gradients(zip(grads_gen, generator.trainable_variables))

    return d_loss, g_loss

print('Training step compiled.')

## Step 8: Train the DCGAN (20 Epochs)

A fixed `noise_seed` is used to generate a 4×4 image grid after every epoch, allowing us to track visual progress over training.

In [10]:
EPOCHS     = 20
noise_seed = tf.random.normal([16, NOISE_DIM])  # fixed seed for consistent visualization
d_losses   = []
g_losses   = []

def save_images(model, epoch):
    """Generate and save a 4x4 grid of images for a given epoch."""
    predictions = model(noise_seed, training=False)
    fig = plt.figure(figsize=(4, 4))
    for i in range(predictions.shape[0]):
        plt.subplot(4, 4, i + 1)
        plt.imshow((predictions[i] * 127.5 + 127.5).numpy()[:, :, 0], cmap='gray')
        plt.axis('off')
    plt.savefig(f'generated_images/image_epoch_{epoch:02d}.png', bbox_inches='tight')
    plt.close()

def train(dataset, epochs):
    for epoch in range(epochs):
        for image_batch in dataset:
            d_loss, g_loss = train_step(image_batch)

        d_losses.append(float(d_loss))
        g_losses.append(float(g_loss))
        print(f'Epoch {epoch+1:2d}, D Loss: {d_loss:.4f}, G Loss: {g_loss:.4f}')
        save_images(generator, epoch + 1)

    print('Training complete!')

train(dataset, EPOCHS)

NameError: name 'tf' is not defined

## Step 9: Plot Training Losses

A well-trained DCGAN has both losses fluctuating near **1.0** (Nash equilibrium). If Discriminator loss → 0, the Generator is failing to improve.

In [13]:
epochs_range = range(1, EPOCHS + 1)

plt.figure(figsize=(10, 5))
plt.plot(epochs_range, d_losses, label='Discriminator Loss', color='red',  marker='o', linewidth=2)
plt.plot(epochs_range, g_losses, label='Generator Loss',     color='blue', marker='s', linewidth=2)
plt.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='Equilibrium (1.0)')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss',  fontsize=12)
plt.title('DCGAN Training Losses Over Epochs', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.xticks(epochs_range)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('generated_images/loss_plot.png', dpi=150)
plt.show()
print(f'Final  D Loss: {d_losses[-1]:.4f}')
print(f'Final  G Loss: {g_losses[-1]:.4f}')

NameError: name 'plt' is not defined

## Step 10: Display Final Generated Images (Epoch 20)

In [16]:
predictions = generator(noise_seed, training=False)

fig, axes = plt.subplots(4, 4, figsize=(6, 6))
fig.suptitle('DCGAN Generated Images — After Epoch 20', fontsize=13, fontweight='bold')
for i, ax in enumerate(axes.flatten()):
    ax.imshow((predictions[i] * 127.5 + 127.5).numpy()[:, :, 0], cmap='gray')
    ax.axis('off')
plt.tight_layout()
plt.savefig('generated_images/final_output.png', dpi=150)
plt.show()

NameError: name 'generator' is not defined

## Step 11: Epoch 1 vs Epoch 20 — Image Quality Progression

In [19]:
from PIL import Image

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
fig.suptitle('DCGAN Image Quality Progression', fontsize=14, fontweight='bold')

paths  = ['generated_images/image_epoch_01.png', 'generated_images/image_epoch_20.png']
titles = ['Epoch 1  — Noisy / Unstructured', 'Epoch 20 — Recognizable Digits']

for ax, path, title in zip(axes, paths, titles):
    ax.imshow(Image.open(path))
    ax.set_title(title, fontsize=11)
    ax.axis('off')

plt.tight_layout()
plt.savefig('generated_images/epoch_comparison.png', dpi=150)
plt.show()

NameError: name 'plt' is not defined

## Conclusion

This experiment demonstrated the implementation and training of a **Deep Convolutional Generative Adversarial Network (DCGAN)** for image generation using the MNIST dataset.

**Key Observations:**
- The Generator and Discriminator were trained adversarially — the generator learns to produce realistic digit images while the discriminator learns to distinguish real from fake.
- Both losses hovered near **1.0–1.3**, indicating a balanced adversarial game without either network dominating.
- Generated images improved visibly from random noise at Epoch 1 to recognizable handwritten digits by Epoch 20.
- **BatchNormalization** ensured stable training, **LeakyReLU** prevented dead neurons, and **Conv2DTranspose** enabled spatial upsampling in the Generator.

Overall, the experiment provided practical insight into deep generative models and adversarial training techniques in deep learning.